# **3. NLI Filtering**

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

### huggingface.co Read Token Required

In [ ]:
from huggingface_hub import login
login()

In [ ]:
drive.mount('/content/drive')

BASE_PATH = "/content/drive/MyDrive/commonsenseqa_parpahrase_evaluation"

eval_df = pd.read_csv(f"{BASE_PATH}/eval_df.csv")
print(eval_df.shape)

In [ ]:
def save_df(df, filename):
    path = os.path.join(BASE_PATH, filename)
    df.to_csv(path, index=False)
    print(f"Saved: {path}")

### Load NLI Model

In [ ]:
nli_model_name = "facebook/bart-large-mnli"
nli_tokenizer = AutoTokenizer.from_pretrained(nli_model_name)
nli_model = AutoModelForSequenceClassification.from_pretrained(nli_model_name)
nli_model.eval()

### NLI Scoring Functions + Dataframe

In [ ]:
# Comparison of sentences (original vs. paraphrase) semantic relationship scores
def nli_score(premise, hypothesis):
    inputs = nli_tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = nli_model(**inputs)
    # Probability scores - contradiction, neutral, entailment
    probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]

    labels = ["contradiction", "neutral", "entailment"]
    scores = dict(zip(labels, probs))

    return scores

In [ ]:
def add_nli_scores(eval_df):
    eval_df = eval_df.copy()

    entailments = []
    neutrals = []
    contradictions = []

    originals = eval_df[eval_df["variant_name"] == "original"][["id", "question_text"]]
    original_map = dict(zip(originals["id"], originals["question_text"]))

    for _, row in eval_df.iterrows():
        if row["variant_name"] == "original":
            entailments.append(1.0)
            neutrals.append(0.0)
            contradictions.append(0.0)
            continue

        premise = original_map[row["id"]]
        hypothesis = row["question_text"]

        scores = nli_score(premise, hypothesis)

        entailments.append(scores["entailment"])
        neutrals.append(scores["neutral"])
        contradictions.append(scores["contradiction"])

    eval_df["nli_entailment"] = entailments
    eval_df["nli_neutral"] = neutrals
    eval_df["nli_contradiction"] = contradictions

    eval_df["nli_margin"] = eval_df["nli_entailment"] - eval_df["nli_contradiction"]

    eval_df["nli_label"] = eval_df[
        ["nli_contradiction", "nli_neutral", "nli_entailment"]
    ].idxmax(axis=1).str.replace("nli_", "", regex=False)

    eval_df["semantic_ok"] = (
        (eval_df["variant_name"] == "original") |
        (
            (eval_df["nli_entailment"] >= 0.70) &
            (eval_df["nli_contradiction"] <= 0.15)
        )
    )

    return eval_df

eval_df = add_nli_scores(eval_df)
save_df_to_drive(eval_df, "eval_df_with_nli.csv")

In [ ]:
eval_df['semantic_ok'].value_counts()

### Experiment Datasets

In [ ]:
exp1_df = eval_df[
    (eval_df["paraphrase_level"] == "original") |
    (eval_df["paraphrase_level"] == "light")
].reset_index(drop=True)

print("Experiment 1 rows:", len(exp1_df))

save_df_to_drive(exp1_df, "exp1_df.csv")

In [ ]:
exp1_df_semantic = eval_df[
    (
        (eval_df["paraphrase_level"] == "original") |
        (eval_df["paraphrase_level"] == "light")
    )
    &
    (eval_df["semantic_ok"] == True)
].reset_index(drop=True)

print("Filtered Experiment 1 rows:", len(exp1_df_semantic))

save_df_to_drive(exp1_df_semantic, "exp1_df_semantic.csv")

In [ ]:
exp2_df = eval_df[
    (eval_df["paraphrase_level"] == "original") |
    (eval_df["paraphrase_level"].isin(["medium", "strong"]))
].reset_index(drop=True)

print("Experiment 2 rows:", len(exp2_df))

save_df_to_drive(exp2_df, "exp2_df.csv")

In [ ]:
exp2_df_semantic = eval_df[
    (
        (eval_df["paraphrase_level"] == "original") |
        (eval_df["paraphrase_level"].isin(["medium", "strong"]))
    )
    &
    (eval_df["semantic_ok"] == True)
].reset_index(drop=True)

print("Filtered Experiment 2 rows:", len(exp2_df_semantic))

save_df_to_drive(exp2_df_semantic, "exp2_df_semantic.csv")